In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import sys

In [ ]:
sys.argv = ['-f'] + ["MIC", "-a", "cpu"]

In [ ]:
import os
sys.path.append("/home/pwiesenbach/BertGCN")
os.chdir("/home/pwiesenbach/BertGCN")

In [ ]:
from entry import * 

In [ ]:
import importlib
import entry
import logging
importlib.reload(logging)
importlib.reload(entry)

In [ ]:
from utils import *
import json 
import pandas as pd
import random
from pathlib import Path
import pickle
from transformers import AutoTokenizer
from operator import itemgetter
from collections import Counter
from clinic_datasets import CleanClinicDataset
import pickle
from torch.utils.data import Subset

In [ ]:
dataset_file = Path("data") / f"medindcls_{args.bertmodel}_{args.doclevel}.json"
if not dataset_file.exists():
    print("Creating dataset")
    dataset = CleanClinicDataset(tokenizer=tokenizer, task="MIC", doclevel=args.doclevel, clean=False)
    with open(dataset_file, "wb") as f:
        print(f"Saving dataset under {dataset_file}")
        pickle.dump(dataset, f)
else:
    print(f"Loading dataset from: {dataset_file}")
    with open(dataset_file, "rb") as f:
        dataset = pickle.load(f)

train_len = 1889
val_len = 270
test_len = 540
nb_word = 25297

random.seed(0)
idx = np.arange(len(dataset))
random.shuffle(idx)

train_idx, val_idx, test_idx = (
    idx[: int(len(idx) * 0.7)],
    idx[int(len(idx) * 0.7) : int(len(idx) * 0.8)],
    idx[int(len(idx) * 0.8) :],
)

train_dataset = Subset(dataset, train_idx)
val_dataset = Subset(dataset, val_idx)
test_dataset = Subset(dataset, test_idx)
    
def map_to_idx(x):
    if x < train_len:
        return train_idx[x]
    elif x < val_len + train_len:
        return val_idx[x - train_len]
    else:
        return test_idx[x - val_len - train_len]

In [ ]:
MODELNAME = Path(PRETRAINEDMODEL).stem
if args.data == "MIC":
    DATASET = "med_indication_all_RF_diag"
    if args.testunklar:
        DATASETPATH =  Path("data") / f"ind.{DATASET}_{args.doclevel}_testunklar"
    else:
        DATASETPATH =  Path("data") / f"ind.{DATASET}_{args.doclevel}"
    BERTSAVEDIR = Path(f"models/finetuned/{args.doclevel}")
    if args.testunklar:
        BERTPATH = Path(f"{BERTSAVEDIR}/{MODELNAME}_med_indication_all_RF_diag_testunklar_best.pt")
    else:
        BERTPATH = Path(f"{BERTSAVEDIR}/{MODELNAME}_med_indication_all_RF_diag_best.pt")
elif args.data == "CSC":
    DATASET = "CARDIODE400_main"
    DATASETPATH =  Path("data") / f"ind.{DATASET}"
    BERTPATH = Path("models/finetuned/gbert-base_CARDIODE400_main_best.pt")

adj, features, y_train, y_val, y_test, train_mask, val_mask, test_mask, _, _ = load_corpus(DATASETPATH)
doc_mask = train_mask + val_mask + test_mask
adj.sum()

In [ ]:
first_order_adj = adj @ adj
first_order_adj = first_order_adj.toarray()

In [ ]:
top_n_input = 10
top_input_test_rel_nodes = np.argpartition(first_order_adj[test_mask][:, doc_mask], -top_n_input)[:, -top_n_input:]
top_input_test_rel_nodes = np.vectorize(map_to_idx)(top_input_test_rel_nodes)
top_input_test_rel_nodes.max(), top_input_test_rel_nodes.shape

In [ ]:
input_df = pd.DataFrame(top_input_test_rel_nodes, index=test_idx)

In [ ]:
input_df

In [ ]:
input_df = pd.melt(input_df,  value_name='rel_id', ignore_index=False).drop(["variable"], axis=1).reset_index(names="id")

In [ ]:
input_df

In [ ]:
labels = dataset.LE.inverse_transform([dataset[x]["labels"] for x in input_df.id])
rel_labels = dataset.LE.inverse_transform([dataset[x]["labels"] for x in input_df.rel_id])
input_df["label"] = labels
input_df["rel_label"] = rel_labels

In [ ]:
input_df

In [ ]:
input_source_df = input_df[["id", "label"]].drop_duplicates()
input_target_df = input_df[["rel_id", "rel_label"]].drop_duplicates()
input_node_df = pd.concat([input_source_df, input_target_df.rename(columns={'rel_id':'id', "rel_label": "label"})], axis=0, ignore_index=True).drop_duplicates()

In [ ]:
input_G=nx.from_pandas_edgelist(input_df, "id", 'rel_id')

In [ ]:
input_id_df = input_df[["id", "label"]]
input_rel_df = input_df[["rel_id", "rel_label"]]

In [ ]:
new_columns = ["id", "label"]
input_id_df.columns = new_columns
input_rel_df.columns = new_columns

In [ ]:
input_id_rel_df = pd.concat([input_id_df, input_rel_df], ignore_index=True).drop_duplicates()
input_id2label = dict(zip(input_id_rel_df.id, input_id_rel_df.label))
nx.set_node_attributes(input_G, input_id2label, "label")

In [ ]:
input_id2text = {id: dataset.texts[id] for id in input_id_rel_df.id}
nx.set_node_attributes(input_G, input_id2text, "text")

In [ ]:
input_id2drug = {node: input_G.nodes()[node]["text"].split(" ")[1] for node in input_G.nodes()}
nx.set_node_attributes(input_G, input_id2drug, "drug")

In [ ]:
nx.is_connected(input_G), nx.number_connected_components(input_G)

In [ ]:
components = nx.connected_components(input_G)
largest_component = max(components, key=len)
subgraph = input_G.subgraph(largest_component)
diameter = nx.diameter(subgraph)
print("Network diameter of largest component:", diameter)

In [ ]:
len(components)

In [ ]:
[Counter([input_G.nodes()[x]["drug"] for x in com]).most_common()[0] for com in components]

In [ ]:
[Counter([input_G.nodes()[x]["label"] for x in com]).most_common()[0] for com in components]

In [ ]:
input_triadic_closure = nx.transitivity(input_G)
input_triadic_closure

In [ ]:
input_degree_dict = dict(input_G.degree(input_G.nodes()))
nx.set_node_attributes(input_G, input_degree_dict, 'degree')
input_sorted_degree = sorted(input_degree_dict.items(), key=itemgetter(1), reverse=True)

input_sorted_degree[:10]

In [ ]:
input_betweenness_dict = nx.betweenness_centrality(input_G)
nx.set_node_attributes(input_G, input_betweenness_dict, 'betweenness')
input_sorted_betweenness = sorted(input_betweenness_dict.items(), key=itemgetter(1), reverse=True)

input_sorted_betweenness[:10]

In [ ]:
input_communities = nx.community.greedy_modularity_communities(input_G)
input_modularity_dict = {}
for i, c in enumerate(input_communities):
    for name in c:
        input_modularity_dict[name] = i
nx.set_node_attributes(input_G, input_modularity_dict, 'community')

len(input_communities)

In [ ]:
[Counter([input_G.nodes()[id]["label"] for id in com]).most_common()[0][1]/len(com) for com in input_communities[:10]]

In [ ]:
[Counter([input_G.nodes()[id]["label"] for id in com]).most_common()[0] for com in input_communities[:10]]

In [ ]:
train_count = 0
val_count = 0
test_count = 0

for node in input_G.nodes():
    if node not in test_idx: 
        continue
    for n in input_G.adj[node]:
        if n in train_idx or n in val_idx:
            train_count += 1
        #elif n in val_idx:
        #    val_count += 1
        else:
            test_count += 1

s = train_count + val_count + test_count
train_count/s, val_count/s, test_count/s